# UK Used Car Price Prediction: 04. Modeling dan Evaluasi

Notebook ini membandingkan Ridge dengan beberapa konfigurasi XGBoost. Pemilihan model memakai MAE cross-validation pada data training saja; test set hanya dipakai untuk evaluasi akhir setelah konfigurasi dipilih.

**Input:** `data/processed/cars_clean.csv` dan `models/03_ridge_baseline.joblib` dari tahap 03.  
**Output:** `models/04_best_model.joblib`, `reports/modeling_cv_results.csv`, dan `reports/model_metrics.json`.

Catatan: notebook 03 sudah menampilkan skor baseline pada test set. Jangan gunakan skor test untuk memilih konfigurasi atau tuning; gunakan hasil cross-validation di notebook ini.

## 1. Setup

Jalankan notebook dengan working directory di folder `notebooks/`, sesuai notebook sebelumnya.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.modeling import run_modeling

print("Project:", PROJECT_ROOT)

## 2. Cross-validation dan pemilihan model

Tiga nilai alpha Ridge dan empat konfigurasi XGBoost dibandingkan dengan 3-fold cross-validation. Setiap fold mempelajari preprocessing dari bagian training fold-nya sendiri. Skor MAE dihitung dalam pound setelah prediksi dikembalikan dari skala log.

In [ ]:
cv_results, test_metrics, metrics = run_modeling()

print("Ranking cross-validation (MAE lebih rendah lebih baik):")
display(cv_results)
print("Model terpilih:", metrics["selected_model"])
print(f"MAE cross-validation terbaik: £{metrics['best_cv_mae']:,.2f}")

## 3. Evaluasi akhir pada test set

Model terpilih ditentukan oleh cross-validation training saja. Tabel berikut membandingkan baseline median, Ridge tahap 03, dan model terpilih pada split test 20% yang sama. Metrik utama adalah MAE; RMSE dan R² menjadi pelengkap.

In [ ]:
display(test_metrics.style.format({
    "MAE (£)": "£{:,.0f}",
    "RMSE (£)": "£{:,.0f}",
    "R²": "{:.3f}",
}))

## 4. Model dan metrik yang disimpan

Model terpilih tersimpan terpisah dari baseline agar hasil tahap 03 tetap tersedia. CV ranking dan metrik holdout disimpan untuk dokumentasi dan penggunaan berikutnya.

In [ ]:
model_path = PROJECT_ROOT / "models" / "04_best_model.joblib"
cv_path = PROJECT_ROOT / "reports" / "modeling_cv_results.csv"
metrics_path = PROJECT_ROOT / "reports" / "model_metrics.json"
print("Model:", model_path)
print("CV results:", cv_path)
print("Metrics:", metrics_path)

## Menggunakan model terpilih

Estimator tersimpan menerima DataFrame dengan fitur input yang sama seperti data bersih, tanpa kolom target `price`. Pipeline di dalam model otomatis membuat `car_age` dan menerapkan preprocessing.

In [ ]:
import joblib

best_model = joblib.load(model_path)
sample_cars = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "cars_clean.csv").drop(columns="price").head(5)
predicted_prices = best_model.predict(sample_cars)
pd.DataFrame({"predicted_price (£)": predicted_prices})